# H14: download the USDY transfer timestamps

Runs the data collection for hypothesis H14 (pre-registration addendum A4, minimum: at least 1,000 timestamps over at least 90 days):

* token: Ondo U.S. Dollar Yield (USDY), Ethereum mainnet
* data: the **latest transfer events**, fetched in whole windows of 10 pages (10,000 events); it stops after the first window at which the series spans at least 90 days, otherwise it continues with the next earlier window
* one filter only: **one timestamp per distinct transaction hash** (a swap creates several transfer events)

*(A first attempt with only 6,000 events gave 2,714 transactions over 64.5 days, below the registered minimum, so the window rule was extended. No H14 result had been computed.)*

**How to use:** menu *Runtime, Run all*. When asked, paste your Etherscan API key (the box is hidden; the key is never printed or saved). At the end `usdy_h14_data.zip` downloads. Send that zip back, then delete the API key on etherscan.io/apidashboard.


In [ ]:
from __future__ import annotations
# Cell 2: functions (nothing to edit)
# ---- H14 data fetch: USDY transfer timestamps (pre-registered rules, see PREREGISTRATION.md addendum A4) ----
import csv, datetime as dt, hashlib, json, time, zipfile
from pathlib import Path

CONTRACT = "0x96F6eF951840721AdBF46Ac996b59E0235CB985C"   # Ondo U.S. Dollar Yield (USDY), Ethereum mainnet
CHAINID, OFFSET, PAGES = 1, 1000, 10                      # one window = up to 10,000 latest events (the API maximum)
MIN_TX, MIN_DAYS, MAX_WINDOWS = 1000, 90, 5               # registered minimum: >=1,000 timestamps over >=90 days
BASE = "https://api.etherscan.io/v2/api"
OUT = Path("h14_data")


def page_url(page: int, key: str, endblock: int | None = None) -> str:
    eb = f"&endblock={endblock}" if endblock else ""
    return (f"{BASE}?chainid={CHAINID}&module=account&action=tokentx&contractaddress={CONTRACT}{eb}"
            f"&page={page}&offset={OFFSET}&sort=desc&apikey={key}")


def span_days_of(events: list[dict]) -> tuple[int, float]:
    tx = {r["hash"]: int(r["timeStamp"]) for r in events}
    if len(tx) < 2:
        return len(tx), 0.0
    return len(tx), (max(tx.values()) - min(tx.values())) / 86400


def fetch_pages(get_json, key: str, pause: float = 1.0) -> list[dict]:
    """Stopping rule fixed by the registered minimum: fetch whole windows of 10 pages (10,000 events, latest first);
    stop after the first window at which the distinct-transaction series spans >= 90 days and has >= 1,000 timestamps;
    otherwise continue with the next earlier window (endblock = lowest block seen). The key is never printed or saved."""
    OUT.mkdir(exist_ok=True)
    pages, endblock, n_files = [], None, 0
    for w in range(1, MAX_WINDOWS + 1):
        window = []
        for p in range(1, PAGES + 1):
            data = get_json(page_url(p, key, endblock))
            if data.get("status") == "0" and data.get("message") == "No transactions found":
                print(f"window {w} page {p}: no more transactions"); break
            if data.get("status") != "1" or not isinstance(data.get("result"), list):
                raise RuntimeError(f"window {w} page {p}: status={data.get('status')!r} message={data.get('message')!r} "
                                   f"result={str(data.get('result'))[:120]!r}")
            n_files += 1
            (OUT / f"usdy_w{w}_p{p}.json").write_text(json.dumps(data))   # raw response (contains no key)
            print(f"window {w} page {p}: {len(data['result'])} records")
            window.append(data); pages.append(data)
            time.sleep(pause)
            if len(data["result"]) < OFFSET:
                break
        events = [r for d in pages for r in d["result"]]
        n_tx, days = span_days_of(events)
        print(f"after window {w}: {len(events)} events, {n_tx} distinct transactions, {days:.1f} days")
        if n_tx >= MIN_TX and days >= MIN_DAYS:
            break
        low = min(int(r["blockNumber"]) for d in window for r in d["result"]) if window else None
        if low is None:
            break
        endblock = low                                   # inclusive: the boundary block is re-fetched, duplicates dropped by hash
    return pages


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def build_outputs(pages: list[dict]) -> dict:
    events = [r for d in pages for r in d["result"]]
    seen, rows = set(), []
    for r in events:                                   # the ONLY filter: one timestamp per distinct transaction hash
        if r["hash"] in seen:
            continue
        seen.add(r["hash"])
        rows.append((int(r["timeStamp"]), int(r["blockNumber"])))
    rows.sort()
    with open(OUT / "timing.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["timestamp", "block"])
        w.writerows(rows)
    t0, t1 = rows[0][0], rows[-1][0]
    span_days = (t1 - t0) / 86400
    iso = lambda t: dt.datetime.fromtimestamp(t, dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")  # noqa: E731
    summary = dict(
        contract=CONTRACT, chainid=CHAINID, pages_per_window=PAGES, offset=OFFSET, files=len(pages),
        retrieved_utc=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        events=len(events), distinct_transactions=len(rows), first_utc=iso(t0), last_utc=iso(t1),
        span_days=round(span_days, 2), first_block=rows[0][1], last_block=rows[-1][1],
        meets_preregistered_minimums=bool(len(rows) >= MIN_TX and span_days >= MIN_DAYS),
        rule="latest events in whole windows of 10 pages until >=1,000 timestamps span >=90 days; one timestamp per distinct transaction hash; no other filter",
        sha256={p.name: sha256(p) for p in sorted(OUT.glob("*")) if p.name != "summary.json"})
    (OUT / "summary.json").write_text(json.dumps(summary, indent=1))
    zpath = Path("usdy_h14_data.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.glob("*")):
            z.write(p, p.name)
    return summary


In [ ]:
# Cell 3: run
import json, getpass, requests
from google.colab import files

key = getpass.getpass("Paste your Etherscan API key (hidden): ").strip()

def get_json(url):
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:          # never show the URL: it contains the key
        raise RuntimeError(f"network error: {type(e).__name__}") from None

pages = fetch_pages(get_json, key)
summary = build_outputs(pages)
print(json.dumps({k: v for k, v in summary.items() if k != "sha256"}, indent=1))
if not summary["meets_preregistered_minimums"]:
    print("\nWARNING: fewer than 1,000 timestamps or less than 90 days. Do not use this data; tell the author.")
files.download("usdy_h14_data.zip")
